# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aryan-0018/FlyRank-ML-W1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

Load the FlyRank starter dataset for the baseline signal checks and rule construction. The baseline uses only information available at the decision moment.

In [43]:
!git clone https://github.com/aryan-0018/FlyRank-ML-W1.git 2>/dev/null || true

In [44]:
!ls FlyRank-ML-W1/data/raw

content_refresh_anonymized.csv


In [45]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = "FlyRank-ML-W1/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Unit of analysis: one row = one content page")

Dataset shape: (30000, 44)
Unit of analysis: one row = one content page


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule framing

**Lane:** Refresh / Content Opportunity Scoring

The baseline will prioritise content pages for editorial review when they show a combination of staleness and meaningful search exposure. The purpose is to rank pages for review, not to claim that a page definitely requires a content change.

I will first test two signals that the rule relies on: `days_since_last_update` as the staleness signal and `impressions_90d` as the search-volume signal. Staleness is linked to FlyRank's refresh flags, while search volume is linked to the quick-win logic. The data will determine whether each signal is useful enough to retain.

The rule will use one score, one reason code, and one action label. It will use only information available at the decision moment and will not use future-window outcomes or label-derived fields.

### Signal check 1 — Staleness

I will bucket `days_since_last_update` into three interpretable groups: 0–30 days, 31–90 days, and more than 90 days. This tests whether older content is meaningfully represented in the data before using staleness in the baseline rule.

**Verdict:** I will assign CONFIRMED, OPPOSITE, MIXED, or FALSE only after inspecting the bucket counts and the relevant observed outcome pattern.

In [46]:
staleness_check = (
    df.assign(
        staleness_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[-1, 30, 90, np.inf],
            labels=["0-30 days", "31-90 days", ">90 days"]
        )
    )
    .groupby("staleness_bucket", observed=False)
    .size()
    .reset_index(name="n")
)

print("Staleness bucket table:")
display(staleness_check)

print("Total n:", staleness_check["n"].sum())

Staleness bucket table:


,staleness_bucket,n
0,0-30 days,20480
1,31-90 days,175
2,>90 days,9345


Total n: 30000


**Verdict: CONFIRMED**

The staleness signal is represented by 9,345 pages with more than 90 days since the last update, so there is a substantial stale-content population to prioritise. The 31–90 day bucket is very small at 175 pages, so I will avoid treating it as a separate scoring tier and instead use the clearly represented `>90 days` condition in the baseline rule.

### Signal check 2 — Search volume

I will use `impressions_90d` as the search-volume signal. This is linked to FlyRank's quick-win logic because search exposure helps determine whether an identified content opportunity has sufficient observed demand to justify review.

I will bucket the observed 90-day impression volume into three groups and inspect the resulting distribution before deciding whether search volume should contribute to the baseline score.

**Verdict:** I will assign CONFIRMED, OPPOSITE, MIXED, or FALSE only after inspecting the observed bucket table.

In [47]:
volume_check = (
    df.assign(
        volume_bucket=pd.qcut(
            df["impressions_90d"],
            q=3,
            labels=["Low", "Medium", "High"],
            duplicates="drop"
        )
    )
    .groupby("volume_bucket", observed=False)
    .size()
    .reset_index(name="n")
)

print("Search-volume bucket table:")
display(volume_check)

print("Total n:", volume_check["n"].sum())

Search-volume bucket table:


,volume_bucket,n
0,Low,10003
1,Medium,9997
2,High,10000


Total n: 30000


**Verdict: CONFIRMED**

The search-volume signal has broad observed variation across the dataset, with approximately 10,000 pages in each quantile bucket. Because these buckets were created using quantiles, the balanced counts demonstrate coverage across the signal rather than predictive strength. I will therefore use impressions_90d as a three-level scoring component based on its observed distribution.

### Baseline rule

The baseline score combines two decision-time signals: staleness and search exposure.

- Staleness contributes 2 points when `days_since_last_update > 90`, 1 point for 31–90 days, and 0 points for 30 days or fewer.
- Search exposure contributes 2 points for the highest third of `impressions_90d`, 1 point for the middle third, and 0 points for the lowest third.
- The total score ranges from 0 to 4 and is used only to rank pages for editorial review.
- The reason code identifies the dominant reason for the score: `STALE_HIGH_VOLUME`, `STALE`, `HIGH_VOLUME`, or `LOW_PRIORITY`.
- The action label is `PRIORITISE_REVIEW`, `REVIEW`, or `MONITOR`.

The rule uses only decision-time fields and contains no future-window or label-derived inputs.

In [48]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [49]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the baseline action score using decision-time signals only.

queue = df.copy()

# Staleness component
queue["staleness_score"] = np.select(
    [
        queue["days_since_last_update"] > 90,
        queue["days_since_last_update"].between(31, 90, inclusive="both")
    ],
    [2, 1],
    default=0
)

# Search-volume component using the observed distribution.
volume_q33 = queue["impressions_90d"].quantile(1/3)
volume_q67 = queue["impressions_90d"].quantile(2/3)

queue["volume_score"] = np.select(
    [
        queue["impressions_90d"] >= volume_q67,
        queue["impressions_90d"] >= volume_q33
    ],
    [2, 1],
    default=0
)

# Combined baseline score.
queue["score"] = queue["staleness_score"] + queue["volume_score"]

# One reason code.
queue["reason_code"] = np.select(
    [
        (queue["staleness_score"] == 2) & (queue["volume_score"] == 2),
        (queue["staleness_score"] == 2),
        (queue["volume_score"] == 2)
    ],
    [
        "STALE_HIGH_VOLUME",
        "STALE",
        "HIGH_VOLUME"
    ],
    default="LOW_PRIORITY"
)

# One action label.
queue["action"] = np.select(
    [
        queue["score"] == 4,
        queue["score"].isin([2, 3])
    ],
    [
        "PRIORITISE_REVIEW",
        "REVIEW"
    ],
    default="MONITOR"
)

# Rank highest-priority pages first.
queue = queue.sort_values(
    ["score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

print("Ranked queue created.")
print("Rows:", len(queue))
print("Score range:", queue["score"].min(), "to", queue["score"].max())

display(
    queue[
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

Ranked queue created.
Rows: 30000
Score range: 0 to 4


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action
0,1,content_5fe46e04994d,104,517715,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
1,2,content_2dba2b1f9536,104,443434,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
2,3,content_2c2606c5d176,104,347399,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
3,4,content_cb112fce36be,104,309910,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
4,5,content_9532f197bbc8,104,309192,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
5,6,content_36ff89c8214e,104,295097,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
6,7,content_b28d1efd668f,104,286608,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
7,8,content_813e88069237,104,233561,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
8,9,content_c21024970297,104,211366,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
9,10,content_c8e9d6ab9013,104,208678,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW


In [50]:
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print(f"Baseline queue written to: {output_path}")
print(f"Rows written: {len(queue)}")
print("Action distribution:")
display(queue["action"].value_counts().rename_axis("action").reset_index(name="n"))

print("\nReason-code distribution:")
display(queue["reason_code"].value_counts().rename_axis("reason_code").reset_index(name="n"))

print("\nScore distribution:")
display(queue["score"].value_counts().sort_index().rename_axis("score").reset_index(name="n"))

Baseline queue written to: work/outputs/baseline_action_score.csv
Rows written: 30000
Action distribution:


,action,n
0,MONITOR,14778
1,REVIEW,10999
2,PRIORITISE_REVIEW,4223



Reason-code distribution:


,reason_code,n
0,LOW_PRIORITY,14878
1,HIGH_VOLUME,5777
2,STALE,5122
3,STALE_HIGH_VOLUME,4223



Score distribution:


,score,n
0,0,8220
1,1,6558
2,2,7573
3,3,3426
4,4,4223


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect the actual top 20 rows selected by the baseline rule.

top20 = queue.head(20).copy()

top20_review = top20[
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "score",
        "reason_code",
        "action"
    ]
].copy()

print("Top-20 baseline review:")
display(top20_review)

print("Top-20 rows:", len(top20_review))

Top-20 baseline review:


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action
0,1,content_5fe46e04994d,104,517715,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
1,2,content_2dba2b1f9536,104,443434,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
2,3,content_2c2606c5d176,104,347399,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
3,4,content_cb112fce36be,104,309910,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
4,5,content_9532f197bbc8,104,309192,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
5,6,content_36ff89c8214e,104,295097,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
6,7,content_b28d1efd668f,104,286608,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
7,8,content_813e88069237,104,233561,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
8,9,content_c21024970297,104,211366,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW
9,10,content_c8e9d6ab9013,104,208678,4,STALE_HIGH_VOLUME,PRIORITISE_REVIEW


Top-20 rows: 20


### Top-20 review

The top 20 pages are all ranked with a score of 4 because they meet both baseline conditions: more than 90 days since the last update and high 90-day search exposure. The ranking within this tied group is determined by `impressions_90d`, with higher observed exposure ranked first.

| Rank | Action | Reason code | Confidence note | What would make it wrong |
|---:|---|---|---|---|
| 1 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 517,715 impressions_90d. | The page may have been intentionally left unchanged, or the impressions may not reflect current demand. |
| 2 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 443,434 impressions_90d. | The content may still be accurate and useful despite its age. |
| 3 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 347,399 impressions_90d. | High exposure does not by itself mean the content needs updating. |
| 4 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 309,910 impressions_90d. | The observed exposure may be stable without requiring editorial action. |
| 5 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 309,192 impressions_90d. | The page could remain correct and relevant despite its age. |
| 6 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 295,097 impressions_90d. | Search exposure alone cannot establish that a refresh would improve performance. |
| 7 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 286,608 impressions_90d. | The page may not have a meaningful content-quality problem. |
| 8 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 233,561 impressions_90d. | The content could be intentionally evergreen and not require revision. |
| 9 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 211,366 impressions_90d. | High impressions may not translate into a useful refresh opportunity. |
| 10 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 208,678 impressions_90d. | The page may already satisfy its search intent despite being old. |
| 11 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 205,915 impressions_90d. | There may be no substantive information that needs updating. |
| 12 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 201,584 impressions_90d. | The baseline does not observe content quality or editorial necessity directly. |
| 13 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 201,111 impressions_90d. | The page may be performing adequately without intervention. |
| 14 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 192,205 impressions_90d. | The page's age may not indicate actual staleness of its information. |
| 15 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 190,623 impressions_90d. | High search exposure does not establish that refreshing the page is beneficial. |
| 16 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 187,893 impressions_90d. | The page may be accurate, evergreen, or intentionally unchanged. |
| 17 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 181,574 impressions_90d. | The observed signals may not correspond to an actionable content opportunity. |
| 18 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 181,514 impressions_90d. | A review may find that no meaningful update is required. |
| 19 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 179,002 impressions_90d. | The page may have high exposure without needing editorial intervention. |
| 20 | PRIORITISE_REVIEW | STALE_HIGH_VOLUME | Strong baseline fit: 104 days stale and 176,296 impressions_90d. | The baseline cannot distinguish genuine refresh opportunities from simply old, high-exposure pages. |

**Review conclusion:** The top-20 queue is internally consistent with the rule. However, these are **decision-support candidates, not confirmed refresh opportunities**. The main weakness is that the baseline uses only age and search exposure, so a human review could find that some high-ranked pages are evergreen, already accurate, or otherwise unsuitable for refresh.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The reviewed weak picks are pages ranked around 7,645–7,649 with a score of 3 and the `STALE` reason code. Each has 104 days since the last update but only 205–206 impressions in the observed 90-day window.

These are plausible weak picks because the baseline gives substantial weight to staleness even when search exposure is relatively modest. A human reviewer could reasonably decide that these pages are lower-value refresh opportunities despite their age.

### Leakage check

The baseline uses only two decision-time inputs:

- `days_since_last_update`
- `impressions_90d`

The source-column check found no columns containing obvious label, outcome, future-window, product-flag, or decision-field keywords. The scoring code also does not reference a future outcome or a label.

Therefore, based on the checks performed in this notebook, there is no observed evidence that future-window outcomes, labels, or existing product decision flags leaked into the baseline score.

The baseline should therefore be treated as a simple decision-support rule rather than a validated predictive model.

In [52]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Identify potentially weak picks and check that the baseline
# does not rely on future outcomes, labels, or product decision flags.

weak_picks = queue[
    (queue["score"] >= 3)
].tail(5)[
    [
        "rank",
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "score",
        "reason_code",
        "action"
    ]
]

print("Example weak/high-priority picks to review:")
display(weak_picks)

# Inspect column names for fields that could represent outcomes,
# labels, future windows, or existing product decisions.
suspicious_keywords = [
    "label", "outcome", "future", "decline",
    "flag", "decision", "score"
]

suspicious_columns = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in suspicious_keywords)
]

print("\nPotentially sensitive columns found in source data:")
print(suspicious_columns)

print("\nBaseline feature inputs used:")
print([
    "days_since_last_update",
    "impressions_90d"
])

Example weak/high-priority picks to review:


,rank,content_id,days_since_last_update,impressions_90d,score,reason_code,action
7644,7645,content_4464b509b8dc,104,206,3,STALE,REVIEW
7645,7646,content_0ceecbd531c7,104,205,3,STALE,REVIEW
7646,7647,content_ef9726f832cb,104,205,3,STALE,REVIEW
7647,7648,content_213627f5853e,104,205,3,STALE,REVIEW
7648,7649,content_7e521d1dd40a,104,205,3,STALE,REVIEW



Potentially sensitive columns found in source data:
[]

Baseline feature inputs used:
['days_since_last_update', 'impressions_90d']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.